## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [1]:
# This is Cell #1




import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [2]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [3]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

# sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [4]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100
#sequence = read_file('warandpeace.txt')

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [5]:
# This is Cell #5

#TODO: Create a list of unique characters from the text sequence
vocab = sorted(list(set(sequence)))

#TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}


#TODO: Convert the entire text based data into numerical data
data = [char_to_idx[char] for char in sequence]


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [6]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [7]:
# This is Cell #7

#TODO: Set your model's hyperparameters

# Hyperparameters
sequence_length = 100    # Length of each input sequence
stride = 3              # Stride for creating sequences
embedding_dim = 128     # Dimension of character embeddings
hidden_size = 256       # Number of features in hidden state
learning_rate = 0.001   # Learning rate for optimizer
num_epochs = 10       # Number of training epochs
batch_size = 64         # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)

After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
> Here
> 
sequence_length
- Original Value: 1000
- New Value: 100
- Meaning: This defines the length of each input sequence fed into the model. Shorter sequences (like 100) may help the model learn more easily by focusing on local context rather than trying to optimize over lengthy sequences.
- Change: Reducing the sequence length can help speed up training and reduce the complexity of learning while still maintaining sufficient context for prediction tasks.

stride
- Original Value: 10
- New Value: 3
- Meaning: The stride determines the number of characters to move forward when creating overlapping sequences from the dataset. A smaller stride results in more overlapping sequences.
- Change: Decreasing the stride will create more training examples by generating more overlapping sequences. This can be beneficial in helping the model learn patterns from the dataset more effectively by providing more data points for training.

embedding_dim
- Original Value: 2
- New Value: 128
- Meaning: This sets the dimensionality for the embedding space where the characters are represented. Higher dimensions allow for more expressive representations of characters.
- Change: Increasing the embedding dimension to 128 is likely to improve the model’s ability to learn complex relationships between characters. 

hidden_size
- Original Value: 1
- New Value: 256
- Meaning: Hidden size defines the number of units in the hidden layer of the RNN. This parameter governs the learning capacity of the model.
- Change: Increasing the hidden size to 256 greatly enhances the model's capacity to learn and remember information from the sequences. 

learning_rate
- Original Value: 200
- New Value: 0.001
- Meaning: The learning rate controls how much to change the model parameters during optimization for each update. It affects convergence speed and stability.
- Change: The original value of 200 is excessively high and likely to cause divergence during training. A learning rate of 0.001 is more suitable, allowing stable convergence.

num_epochs
- Original Value: 1
- New Value: 10, (for warandpeace.txt I went back to 1)
- Meaning: This sets how many times the entire training dataset will be passed through the model. More epochs allow the model to learn better from the data.
- Change: Increasing to 10 epochs gives the model more opportunity to learn from the dataset. This is especially important when using very complex datasets

batch_size
- Original Value: 64
- New Value: 64 (No change)
- Meaning: This parameter specifies how many training samples are processed at once during optimization.


vocab_size, input_size, output_size
- Meaning: They ensure that the model is correctly sized to handle input and output for character prediction tasks.

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [8]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]

# Print the sizes of the datasets for verification
print("Training Data Size:", train_data.size())
print("Testing Data Size:", test_data.size())


Training Data Size: torch.Size([2340])
Testing Data Size: torch.Size([260])


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [9]:
# This is Cell #9
train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

# Keep drop_last=True for training to ensure consistent batch sizes
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
# Set drop_last=False for testing to keep all samples
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

total_batches = len(train_loader)


## Defining the RNN Model

Here we will define our character-level RNN model.

In [10]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [11]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size, hidden_size, output_size, embedding_dim).to(device)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [12]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/10:   0%|          | 0/11 [00:00<?, ?it/s]

/tmp/ipykernel_68679/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_68679/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/10: 100%|██████████| 11/11 [00:02<00:00,  4.36it/s]


Epoch [1/10], Loss: 2.0361, Accuracy: 89.15%


Epoch 2/10: 100%|██████████| 11/11 [00:01<00:00,  7.12it/s]


Epoch [2/10], Loss: 0.1826, Accuracy: 99.78%


...

Epoch [8/10], Loss: 0.0104, Accuracy: 99.51%


Epoch 9/10: 100%|██████████| 11/11 [00:01<00:00, 10.36it/s]


Epoch [9/10], Loss: 0.0088, Accuracy: 99.62%


Epoch 10/10: 100%|██████████| 11/11 [00:01<00:00, 10.23it/s]

Epoch [10/10], Loss: 0.0070, Accuracy: 99.78%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [13]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [14]:
# This is Cell #15
with torch.no_grad():
    model.eval()
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_inputs, batch_targets in test_loader:
        current_batch_size = batch_inputs.size(0)
        
        # Reinitialize hidden state if batch size changes
        if current_batch_size != batch_size:
            hidden = model.init_hidden(current_batch_size)
            
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)
        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))

        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)
        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)

        total_loss += loss.item()

    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")



Test Loss: 0.0021, Test Accuracy: 100.00%


/tmp/ipykernel_68679/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_68679/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [ ]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    model.eval()  # Set the model to evaluation mode
    start_text = start_text.lower()
    generated_text = start_text  # Initialize generated text with the starting text

    # Initialize hidden state
    hidden = model.init_hidden(1).to(device)  # batch size of 1 for generation

    with torch.no_grad():  # No need to compute gradients for text generation
        for _ in range(k):
            # Convert last character to input tensor
            current_char = generated_text[-1]
            input_idx = torch.tensor([[char_to_idx[current_char]]]).to(device)
            
            # Forward pass through the model
            output, hidden = model(input_idx, hidden)
            
            # Get the predictions for the next character
            next_char_logits = output[0]  # Shape [1, vocab_size]
            
            # Sample from the output distribution
            next_char_index = sample_from_output(next_char_logits, temperature)
            
            # Convert the predicted index to character and add to generated text
            next_char = idx_to_char[next_char_index.item()]
            generated_text += next_char

    return generated_text

# Interactive loop
print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.


## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.


`"abcdefghijklmnopqrstuvwxyz" * 100`  had a very low final loss value of 0.0070 and had an accuracy value of 99.78. It was also able to run very quickly while calculating the respective loss and accuracy values. The generated text would just continue on the unputed words with order of the alphabet after the last character. For example input for initial text "hello", number of character to generate: 5, temperature equal to 1 would output the sequence: hellopqrst. Changing the temperature value to 2 however would output a sequence of an ordered number of aplahbets which would not start from the last character of the input, for example "helloabcdpqrs...". For temperature value lower than 1, it would just give the same output as if it were 1. A high temperature value such as 3 would just generate a random sequence of letters.


 `warandpeace.txt` had a loss of 1.4389 and an accuracy of 54.41% It took extremely long to run so I had to change my num_epochs value to 1 as for one it around 20 minutes to run, with a higher value such as 10 it would take around 200 minutes so I stuck to one. With a temperature value of 1 I got mostly real words with spaces however the sequence of the words didn't make sense. With a higher temperature value such as 2, the words were incorrect as well as it was generating words that do not exist and are gramitcally incorrect. With a lower value such as 0.5 not only were the words correct but the order of the words made sense. 

 The biggest challenge I faced was to see the model's accuracy and loss with `warandpeace.txt`. This made it really hard to test my code as I had to wait 20 minutes to run the function that trained based on the dataset each time. Key insights I gained about RNNs and sequence modeling are that it's a long and complex task, however, it is a very interesting process and I am fascinated by it!